## 1. Objectif du projet
Le but est de concevoir un classifieur d’images capable d’**identifier automatiquement les photos** parmi plusieurs types d’images (dessins, peintures, schémas, textes).

Deux formulations de la tâche :
- **Classification multi-classes** : distinguer 5 types d’images (`Painting`, `Photo`, `Schematics`, `Sketch`, `Text`).
- **Classification binaire** : identifier si une image est une **photo** (`Photo`) ou **non-photo** (toutes les autres).

## 2. Données utilisées
- Les images ont été réparties dans des dossiers distincts selon leur catégorie.
- Les labels sont extraits automatiquement à partir des noms de répertoire.
- Taille des jeux :
  - Environ 1000 images par classe
  - Split en **train / val / test** avec `ImageDataGenerator` ou `image_dataset_from_directory`

## 3. Stratégie expérimentale
Nous avons construit **12 variantes** selon :
- Le type de tâche : `multi` vs `bin`
- Le type de modèle : `CNN` (from scratch) vs `transfer` (VGG16)
- Les techniques de régularisation : `class_weight`, `fine_tuning`

**Exemples :**
- `Leyanda_CNN_bin_e10_b64`
- `Leyanda_transfer_multi_e10_b64_fine_tuning`
- `Leyanda_transfer_bin_e10_b64_class_weight_fine_tuning`


## 4. Architectures utilisées

### a. Modèle CNN from scratch :
Nous avons construit un modèle simple de classification binaire avec l'architecture suivante :

```python
model = Sequential([
    Rescaling(1./255),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])
```

![Archi Custom.drawio.png](../assets/L1/ArchiCustom.drawio.png)
![Archi Custom Binary.drawio.png](../assets/L1/ArchiCustom%20Binary.drawio.png)


### b. Modèle en transfert d’apprentissage :
Nous avons opté pour l'utilisation de **VGG16 pré-entraîné sur ImageNet** comme base pour le transfert learning, avec l'option `include_top=False` :

```python
base_model = VGG16(include_top=False, weights='imagenet')
```

L'architecture est la suivante :

- Data Augmentation (rotation, zoom, translation, contraste, luminosité)
- VGG16 (non entraînable ou partiellement dégélé selon les tests)
- GlobalAveragePooling2D
- Dense(128, activation='relu')
- Dense(1, activation='sigmoid') pour la sortie binaire

Cette architecture a montré de meilleures performances sur notre tâche de classification entre photos et autres types d’images (dessins, peintures...).

![Archi Custom Transfer Learning.drawio.png](../assets/L1/ArchiTransfer%20Learning.drawio.png)
![Archi Custom Transfer Learning Binary.drawio.png](../assets/L1/ArchiTransfer%20Learning%20Binary.drawio.png)

## 5. Paramètres d'entraînement
- **Optimiseur** : Adam, `learning_rate=0.001`
- **Fonction de perte** :
  - `binary_crossentropy` (binaire)
  - `categorical_crossentropy` (multi-classes)
- **Batch size** : 64
- **Epochs** : 10 (avec early stopping sur `val_loss`)
- **Callbacks** :
  - `ModelCheckpoint` (sauvegarde du meilleur modèle)
  - `ReduceLROnPlateau`
  - `EarlyStopping(patience=3)`

## 6. Méthodes d’évaluation
Chaque modèle est évalué selon :
- Accuracy globale (train/val/test)
- Loss globale
- Matrice de confusion (visuelle, par classe)
- Analyse biais/variance
- Suivi des courbes `loss` et `accuracy` sur 10 epochs

## 7. Objectifs spécifiques analysés
Nous avons notamment exploré :
- Comment les modèles **pré-entraînés** se comparent à un CNN simple
- L’intérêt du **class weighting** face à des déséquilibres (ex. peu de photos réalistes)
- L'effet du **fine tuning** en deuxième phase d'entraînement
- L’efficacité du **formalisme binaire** pour isoler les `photos`

## 8. Analyse des résultats


| Modèle                                                  | Type                                               | Accuracy | Loss   | Commentaires                                                                                                                                         |
|----------------------------------------------------------|----------------------------------------------------|----------|--------|------------------------------------------------------------------------------------------------------------------------------------------------------|
| Leyanda_CNN_multi_e10_b64                                | Multi-class                                        | 0.8430   | 0.4387 | Bonne performance globale. Confusions notables entre 'Painting' et 'Photo', 'Painting' et 'Schematics'.                                             |
| Leyanda_CNN_multi_transfer_e10_b64                       | Multi-class                                        | 0.9339   | 0.2094 | Très bonne performance. Forte amélioration avec le transfert learning. Faibles confusions entre les classes.                                       |
| Leyanda_CNN_multi_transfer_e10_b64_class_weight          | Multi-class (class weight)                         | 0.9294   | 0.2245 | Très bon équilibre. Légère dégradation par rapport au modèle sans pondération, mais meilleures performances sur classes minoritaires.              |
| Leyanda_CNN_bin_Photo_e10_b64_class_weight               | Binaire (Photo vs Non-Photo, class weight)         | 0.8292   | 0.3323 | Bonne précision sur la classe 'Photo'. Déséquilibre visible : taux de faux positifs non négligeable sur les Non-Photo.                             |
| Leyanda_CNN_bin_Photo_e10_b64                            | Binaire (Photo vs Non-Photo)                       | 0.8868   | 0.2498 | Bonne précision globale. Moins de faux positifs que le modèle pondéré, mais légèrement plus de faux négatifs sur les photos.                       |
| Leyanda_CNN_bin_Photo_transfer_e10_b64                   | Binaire (Photo vs Non-Photo, transfer learning)    | 0.9563   | 0.1215 | Excellente performance avec transfert learning. Très peu d'erreurs, surtout sur la classe 'Photo'.                                                  |
| Leyanda_CNN_bin_Photo_transfer_e10_b64_fine_tuning       | Binaire (Photo vs Non-Photo, transfer + fine-tune) | 0.9060   | 0.2323 | Fine-tuning dégrade légèrement les performances par rapport au simple transfert learning. Forte hausse des faux négatifs.                          |
| Leyanda_CNN_bin_Photo_transfer_e10_b64_class_weight_fine_tuning | Binaire (Photo vs Non-Photo, transfer + class weight + fine-tune) | 0.8862 | 0.2651 | Baisse importante des performances sur les photos (faux négatifs élevés). Le fine-tuning semble trop agressif ici.                                 |
| Leyanda_CNN_bin_Photo_transfer_e10_b64_class_weight      | Binaire (Photo vs Non-Photo, transfer + class weight) | 0.9542 | 0.1513 | Très bon équilibre entre précision et rappel. L’ajout du class weight améliore encore la performance sans fine-tuning.                             |
| Leyanda_CNN_multi_e10_b64_class_weight                   | Multi-class (class weight)                         | 0.7357   | 0.7170 | Résultats globalement plus faibles. Pondération seule n’a pas suffi à corriger les fortes confusions, notamment entre Paintings, Photos et Schematics. |
| Leyanda_CNN_multi_transfer_e10_b64_class_weight_fine_tuning | Multi-class (transfer + class weight + fine-tuning) | 0.2452 | 1.5996 | Erreur pendant l'entraînement. Le modèle prédit exclusivement la classe 'Painting'. Résultat invalide à ignorer.                                   |


## Conclusion sur les performances des modèles

Parmi l’ensemble des modèles évalués dans le cadre du projet **Leyanda**, deux modèles se sont particulièrement distingués :

### Meilleur modèle
- **Modèle :** `Leyanda_CNN_bin_Photo_transfer_e10_b64`
- **Type :** Binaire (Photo vs Non-Photo, transfer learning)
- **Accuracy :** 0.9563
- **Loss :** 0.1215
- **Forces :** Ce modèle offre un excellent compromis entre précision et généralisation, avec très peu d’erreurs sur les deux classes. Le transfert learning permet une convergence rapide et efficace, sans avoir recours à des ajustements supplémentaires (comme les poids de classes ou le fine-tuning).
- **Remarque :** Il est simple, stable et performant — idéal pour une mise en production.

### Deuxième meilleur modèle
- **Modèle :** `Leyanda_CNN_bin_Photo_transfer_e10_b64_class_weight`
- **Type :** Binaire (Photo vs Non-Photo, transfer + class weight)
- **Accuracy :** 0.9542
- **Loss :** 0.1513
- **Forces :** Très proche du meilleur modèle, il introduit une pondération des classes qui améliore la prise en compte des déséquilibres sans nuire à la performance globale. Il est légèrement moins performant en loss, mais reste très robuste.

### Conclusion générale

Les modèles **binaires** surpassent globalement les modèles **multi-classes** en termes de précision, en particulier lorsqu’ils utilisent le **transfert learning**. Pour une tâche de tri automatisé entre photos et autres types d’images, les versions **sans fine-tuning** mais **avec transfert learning** offrent les meilleurs résultats, tout en restant simples à maintenir et à déployer.
